# 04 - Baseline ANN

Trains the baseline ANN (128-64-32, ReLU, softmax) on scaled geometric features only (**no uncertainty signals at all**)

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.utils import set_global_seed, load_config, get_logger, configure_gpu_memory_growth
configure_gpu_memory_growth()  # must run before any GPU-touching call (build_model, tf.one_hot, etc.)
from data.paths import get_dataset_config, get_split_file, get_checkpoints_dir, get_results_dir
from src.classes import get_class_registry
from src.features import DEFAULT_FEATURE_NAMES
from src.losses import get_sparse_categorical_crossentropy_loss
from src.model import build_model, compile_model, train_model, make_tf_dataset
from src.metrics import compute_metrics, results_to_dict

In [ ]:
DATASET = "ahn3"

config = load_config("default")
set_global_seed(config["experiment"]["seed"])
dataset_cfg = get_dataset_config(DATASET)
class_registry = get_class_registry(DATASET)
num_classes = class_registry.num_classes
feature_columns = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES

In [ ]:
'''
Since the input point clouds, resulting files and the intermediate variables that we will be dealing with are huge in terms of shear size
we wont be keeping them in the memory for better optimization
'''
train_df = pd.read_csv(get_split_file(DATASET, "train"))
test_df  = pd.read_csv(get_split_file(DATASET, "test"))

# float32 to save memory footprint
X_train = train_df[feature_columns].values.astype(np.float32)
y_train = train_df["class_idx"].values.astype(np.int32)
X_test  = test_df[feature_columns].values.astype(np.float32)
y_test  = test_df["class_idx"].values.astype(np.int32)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"num_classes={num_classes}, feature_dim={X_train.shape[1]}")

del train_df, test_df

## Build, compile and train

In [ ]:
model = build_model(
    input_dim=X_train.shape[1],
    num_classes=num_classes,
    config=config,
)
model.summary()

model = compile_model(model, config, loss=get_sparse_categorical_crossentropy_loss())

train_ds = make_tf_dataset(
    X_train, y_train,
    batch_size=config["training"].get("batch_size", 512),
    shuffle=True,
    seed=config["experiment"]["seed"],
)

ckpt_dir = get_checkpoints_dir(DATASET, "baseline")
model, history = train_model(model, train_ds, None, config, checkpoint_path=ckpt_dir / "best.keras")

## Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    axes[0].plot(history.history["val_loss"], label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss"); axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train acc")
if "val_accuracy" in history.history:
    axes[1].plot(history.history["val_accuracy"], label="val acc")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training accuracy"); axes[1].legend()

plt.tight_layout(); plt.show()

## Evaluate on test split

In [ ]:
y_pred = np.argmax(model.predict(X_test, batch_size=config["training"].get("batch_size", 512), verbose=0), axis=1)
metrics = compute_metrics(
    y_test, y_pred,
    num_classes=num_classes,
    class_names=class_registry.class_names_in_order(),
    excluded_classes=dataset_cfg.get("classes_excluded_from_metrics", []),
)

print(f"=== baseline ===")
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"mIoU     : {metrics['mIoU']:.4f}")
print()
print(metrics["per_class_iou"].to_string(index=False))

result = results_to_dict(metrics, model_name="baseline", dataset=DATASET)


## Save

In [ ]:
ckpt_dir = get_checkpoints_dir(DATASET, "baseline")
model.save(str(ckpt_dir / "model.keras"))
print(f"Model saved -> {ckpt_dir}")

res_dir = get_results_dir(DATASET, "baseline")
metrics["per_class_iou"].to_csv(res_dir / "per_class_iou.csv", index=False)
pd.DataFrame([result]).to_csv(res_dir / "summary.csv", index=False)
print(f"Results saved -> {res_dir}")